# Primeiro teste de modelo

In [1]:
import sys
!{sys.executable} -m pip install pandas tensorflow scikit-learn

   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.5 MB 1.7 MB/s eta 0:00:07
   - -------------------------------------- 0.5/11.5 MB 1.7 MB/s eta 0:00:07
   -- ------------------------------------- 0.8/11.5 MB 1.3 MB/s eta 0:00:09
   --- ------------------------------------ 1.0/11.5 MB 1.3 MB/s eta 0:00:09
   ----- ---------------------------------- 1.6/11.5 MB 1.3 MB/s eta 0:00:08
   ----- ---------------------------------- 1.6/11.5 MB 1.3 MB/s eta 0:00:08
   ------ --------------------------------- 1.8/11.5 MB 1.3 MB/s eta 0:00:08
   ------- -------------------------------- 2.1/11.5 MB 1.2 MB/s eta 0:00:08
   -------- ------------------------------- 2.4/11.5 MB 1.2 MB/s eta 0:00:08
   --------- ------------------------------ 2.6/11.5 MB 1.1 MB/s eta 0:00:08
   --------- ------------------------------ 2.6/11.5 MB 1.1 MB/s eta 0:00:08
   ----------


[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: C:\Users\progr\OneDrive\Documentos\Miguel\PROJETOS\Rainha_de_Saba\.venv\Scripts\python.exe -m pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam

# 1. Carregar o dataset
data = pd.read_csv('C:/Users/progr/OneDrive/Documentos/Miguel/PROJETOS/Rainha_de_Saba/01_data/processed/bitcoin_processed.csv')  # Substitua pelo caminho correto do seu arquivo
data

,Data,Último,Abertura,Máxima,Mínima,Vol.,Var%,Média movel curta,Média movel longa,Sinal,...,RSI30,RSI200,%K10,%D10,%K30,%D30,%K200,%D200,simple_rtn,log_rtn
0,2025-04-17,84623.0,84032.2,84999.0,83822.8,"52,98K","0,70%",84623.000000,84623.000000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-04-16,84032.2,83648.1,85438.2,83143.5,"63,97K","0,46%",84327.600000,84327.600000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.006982,-0.007006
2,2025-04-15,83647.0,84586.8,86438.8,83602.7,"62,45K","-1,11%",84100.733333,84100.733333,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.004584,-0.004594
3,2025-04-14,84586.4,83752.8,85794.9,83705.2,"78,03K","1,02%",84222.150000,84222.150000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.011231,0.011168
4,2025-04-13,83734.4,85282.5,85999.5,83049.6,"70,93K","-1,83%",84124.600000,84124.600000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.010073,-0.010124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1093,2022-04-20,41368.0,41499.0,42203.0,40915.0,"382,84M","-0,33%",39655.300000,35325.376667,1.0,...,65.955071,56.031851,69.568523,63.793893,90.278410,89.199641,93.681552,93.063528,0.021886,0.021650
1094,2022-04-19,41503.0,40809.0,41746.0,40585.0,"268,28M","1,72%",39947.500000,35698.613333,1.0,...,66.122661,56.080762,72.134575,65.138703,91.098157,88.758337,94.214338,92.693595,0.003263,0.003258
1095,2022-04-18,40803.0,39700.0,41095.0,38577.0,"484,26M","2,77%",40053.000000,36102.590000,1.0,...,64.421619,55.757528,58.829120,66.844073,86.847618,89.408062,91.451743,93.115878,-0.016866,-0.017010
1096,2022-04-17,39703.0,40382.0,40599.0,39561.0,"210,01M","-1,68%",40099.000000,36411.440000,1.0,...,61.835659,55.254556,37.920547,56.294748,80.168200,86.037992,87.110524,90.925535,-0.026959,-0.027329


In [6]:
# 2. Selecionar a coluna de preços (ajuste conforme necessário)
prices = data['Último'].values  # Aqui usamos o preço de fechamento, mas pode incluir outros indicadores

# 3. Normalizar os dados para o intervalo [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
prices_scaled = scaler.fit_transform(prices.reshape(-1, 1))

# 4. Criar sequências de dados (Janela de tempo)
def create_sequences(data, time_step=60):
    X, y = [], []
    for i in range(len(data) - time_step):
        X.append(data[i:i + time_step])
        y.append(data[i + time_step])
    return np.array(X), np.array(y)

time_step = 60  # Usar 60 dias para prever o próximo dia
X, y = create_sequences(prices_scaled, time_step)

# 5. Dividir os dados em treino e teste
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

In [7]:
# 6. Modelar a LSTM
model = Sequential()

# Camada LSTM com 50 unidades
model.add(LSTM(units=50, return_sequences=False, input_shape=(X_train.shape[1], X_train.shape[2])))

# Camada densa (fully connected) para prever o valor de saída
model.add(Dense(units=1))

# 7. Compilar o modelo
model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# 8. Treinar o modelo
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))

# 9. Previsão
predictions = model.predict(X_test)

# 10. Inverter a normalização para obter os valores reais
predictions = scaler.inverse_transform(predictions)
y_test = scaler.inverse_transform(y_test.reshape(-1, 1))

C:\Users\progr\OneDrive\Documentos\Miguel\PROJETOS\Rainha_de_Saba\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 7s 71ms/step - loss: 0.1372 - val_loss: 0.0083
Epoch 2/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - loss: 0.0066 - val_loss: 0.0029
Epoch 3/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 47ms/step - loss: 0.0020 - val_loss: 6.9630e-04
Epoch 4/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - loss: 0.0011 - val_loss: 6.4769e-04
Epoch 5/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - loss: 0.0011 - val_loss: 6.2734e-04
Epoch 6/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - loss: 0.0011 - val_loss: 6.0376e-04
Epoch 7/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 45ms/step - loss: 0.0011 - val_loss: 5.7934e-04
Epoch 8/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - loss: 0.0010 - val_loss: 5.7877e-04
Epoch 9/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step - loss: 9.6092e-04 - val_loss: 5.6878e-04
Epoch 10/10
26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - loss: 9.5831e-04 - val_loss: 5.2917e-04
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 86ms/step


In [8]:
# 11. Exibir os resultados
print("Previsões:", predictions[:10])
print("Valores reais:", y_test[:10])

Previsões: [[17740.072]
 [17796.246]
 [17840.457]
 [18005.275]
 [18316.615]
 [18724.02 ]
 [19180.535]
 [19618.504]
 [19986.207]
 [20291.861]]
Valores reais: [[15886.9]
 [18527.4]
 [20589. ]
 [20916.3]
 [21301.6]
 [21145.9]
 [20206.4]
 [20154.4]
 [20483.5]
 [20496.3]]
